# Renewable Energy Efficiency Analyzer

This notebook is the main workflow for the REEA project. It imports the project modules, loads the sample renewable energy dataset, calculates yield gaps, and summarizes site performance.

In [ ]:
from pathlib import Path

import pandas as pd

from energy_site import SolarFarm
from utils import data_chunk_generator

## Load the Dataset

The sample dataset contains hourly renewable energy measurements, including actual output and expected output in kilowatts.

In [ ]:
data_path = Path("data/sample_data.csv")

if not data_path.exists():
    raise FileNotFoundError(f"Could not find dataset: {data_path}")

raw_data = pd.read_csv(data_path)
raw_data.head()

## Run Site Analysis

The `SolarFarm` class loads the data, calculates the yield gap, and computes the performance ratio for each row.

In [ ]:
site = SolarFarm("Demo Solar Site", capacity_kw=500)
site.load_data(data_path)

results = site.calculate_yield_gap()
results[["timestamp", "actual_output_kw", "expected_power_kw", "yield_gap_kw", "performance_ratio"]]

## Summarize Results

These summary values show the average performance ratio and average production shortfall across the dataset.

In [ ]:
summary = {
    "site_name": site.name,
    "capacity_kw": site.capacity_kw,
    "rows_analyzed": len(results),
    "average_performance_ratio": round(results["performance_ratio"].mean(), 3),
    "average_yield_gap_kw": round(results["yield_gap_kw"].mean(), 3),
}

summary

## Demonstrate Chunk Loading

The generator function can read larger CSV files in smaller pieces, which supports the project requirement for generator-based processing.

In [ ]:
first_chunk = next(data_chunk_generator(data_path, chunk_size=5))
first_chunk